In [ ]:
import torch

In [ ]:
torch.cuda.is_available()

True

In [ ]:
torch.cuda.get_device_name(0)

'Tesla T4'

In [ ]:
!pip install ultralytics yt-dlp -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.2 MB/s eta 0:00:00


In [ ]:
!wget "https://github.com/intel-iot-devkit/sample-videos/raw/refs/heads/master/car-detection.mp4" -O traffic.mp4
print("Downloaded!", __import__('os').path.getsize("traffic.mp4"), "bytes")

--2026-06-15 13:35:22--  https://github.com/intel-iot-devkit/sample-videos/raw/refs/heads/master/car-detection.mp4
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/refs/heads/master/car-detection.mp4 [following]
--2026-06-15 13:35:23--  https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/refs/heads/master/car-detection.mp4
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2811553 (2.7M) [application/octet-stream]
Saving to: ‘traffic.mp4’

traffic.mp4         100%[===================>]   2.68M  --.-KB/s    in 0.02s   

2026-06-15 13:35:24 (123 MB/s) - 

In [ ]:
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO
import cv2

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
model = YOLO("yolov8n.pt")
vehicle_classes = [2,3,5,7]
labels = {2:"Car", 3:"Motorcycle", 5:"Bus", 7:"Truck"}

In [ ]:
cap = cv2.VideoCapture("traffic.mp4")

In [ ]:
fps = cap.get(cv2.CAP_PROP_FPS)
print(fps)

60.0


In [ ]:
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(width,height)

1280 720


In [ ]:
out = cv2.VideoWriter("output.mp4",cv2.VideoWriter_fourcc(*"mp4v"),fps,(width,height))

In [ ]:
frame_num = 0

In [ ]:
while True:
  ret, frame = cap.read()
  if not ret:
    break
  frame_num += 1
  results = model.track(frame, verbose = False)[0] # Changed from model(frame, persist = True, verbose = False)
  vehicle_count = 0
  active_ids =[]
  for box in results.boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    if cls_id in vehicle_classes and conf >0.5:
      vehicle_count += 1
      x1,y1,x2,y2 = map(int, box.xyxy[0])
      track_id = int(box.id[0]) if box.id is not None else 0
      active_ids.append(track_id)
      label = f"{labels[cls_id]} {conf:.2f}"
      cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
      cv2.putText(frame, label, (x1, y1 - 10),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
  if frame_num % 30 == 0:
        print(f"Frame {frame_num} — Vehicles detected: {vehicle_count} — IDs: {active_ids}")

  out.write(frame)

cap.release()
out.release()
print("\n✅ Done! Saved to output_detected.mp4")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 312ms
Prepared 1 package in 50ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Frame 30 — Vehicles detected: 13 — IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Frame 60 — Vehicles detected: 13 — IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Frame 90 — Vehicles detected: 14 — IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Frame 120 — Vehicles detected: 14 — IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Frame 150 — Vehicles detected: 14 — IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Frame 180 — Vehicles detected: 13 — IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Frame 210 — Vehicles detected: 13 — IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Frame 240 — Vehicles detected: 13 — IDs

In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture("traffic.mp4")

ret, frame = cap.read()
results = model(frame, verbose=False)[0]

print("All detections in frame 1:")
for box in results.boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    class_name = model.names[cls_id]
    print(f"  {class_name} (class {cls_id}) — confidence: {conf:.2f}")

cap.release()

All detections in frame 1:
  car (class 2) — confidence: 0.90
  car (class 2) — confidence: 0.88
  car (class 2) — confidence: 0.86
  car (class 2) — confidence: 0.85
  car (class 2) — confidence: 0.81
  car (class 2) — confidence: 0.74
  car (class 2) — confidence: 0.70
  car (class 2) — confidence: 0.67
  car (class 2) — confidence: 0.66
  car (class 2) — confidence: 0.61
  bus (class 5) — confidence: 0.57
  car (class 2) — confidence: 0.52
  truck (class 7) — confidence: 0.51
  car (class 2) — confidence: 0.49
  car (class 2) — confidence: 0.48
  car (class 2) — confidence: 0.43
  car (class 2) — confidence: 0.43
  car (class 2) — confidence: 0.35
  car (class 2) — confidence: 0.35
  car (class 2) — confidence: 0.35
  car (class 2) — confidence: 0.34
  car (class 2) — confidence: 0.33
  car (class 2) — confidence: 0.31
  truck (class 7) — confidence: 0.31
  car (class 2) — confidence: 0.26
